# 13 · Model Training

**Purpose**: Baseline benchmarks + LightGBM training + model selection

| I/O | Path |
|-----|------|
| Input  | `data/processed/train.parquet` |
| Input  | `data/processed/val.parquet` |
| Input  | `data/processed/feature_manifest.json` |
| Output | `model/lgbm_atd_model.pkl` |
| Output | `model/model_metadata.json` |
| Output | `data/processed/val_predictions.parquet` |

## Model choice
LightGBM with `regression_l1` (MAE-optimised).  Robust to the
long-tail ATD distribution; native categorical support avoids
one-hot explosion on territory / geo_archetype.

---
## 1 · Setup

In [ ]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})
SEED = 42
SLA_THRESHOLD_MIN = 45

TRAIN_PATH    = Path('../data/processed/train.parquet')
VAL_PATH      = Path('../data/processed/val.parquet')
MANIFEST_PATH = Path('../data/processed/feature_manifest.json')
MODEL_DIR     = Path('../model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH    = MODEL_DIR / 'lgbm_atd_model.pkl'
META_PATH     = MODEL_DIR / 'model_metadata.json'
VAL_PRED_PATH = Path('../data/processed/val_predictions.parquet')

---
## 2 · Load Data + Manifest

In [ ]:
with open(MANIFEST_PATH, 'r', encoding='utf-8') as fh:
    manifest = json.load(fh)

NUMERIC_FEATURES = manifest['numeric_features']
CAT_FEATURES     = manifest['cat_features']
TARGET           = manifest['target']

df_train = pd.read_parquet(TRAIN_PATH)
df_val   = pd.read_parquet(VAL_PATH)

# Ensure categorical dtype
for col in CAT_FEATURES:
    for split in [df_train, df_val]:
        if col in split.columns:
            split[col] = split[col].astype('category')

FEATURE_COLS = [
    c for c in NUMERIC_FEATURES + CAT_FEATURES
    if c in df_train.columns
]

print(f'Train rows    : {len(df_train):,}')
print(f'Val rows      : {len(df_val):,}')
print(f'Features      : {len(FEATURE_COLS)}')
print(f'  Numeric     : {len(NUMERIC_FEATURES)}')
print(f'  Categorical : {len(CAT_FEATURES)}')
print(f'Target        : {TARGET}')

---
## 3 · Helper: Metric Functions

In [ ]:
def compute_mape(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    """MAPE excluding zero actuals."""
    mask = y_true > 0
    return float(
        np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100
    )


def compute_sla_rate(
    y_pred: np.ndarray,
    threshold: float = SLA_THRESHOLD_MIN,
) -> float:
    """Fraction of predicted trips at or below SLA threshold."""
    return float(np.mean(y_pred <= threshold) * 100)


def evaluate_model(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label: str = 'Model',
) -> dict:
    """Compute MAE, RMSE, MAPE, R², SLA-hit-rate."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = compute_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    sla_actual    = float(np.mean(y_true <= SLA_THRESHOLD_MIN) * 100)
    sla_predicted = compute_sla_rate(y_pred)
    return {
        'label':          label,
        'MAE':            round(mae, 3),
        'RMSE':           round(rmse, 3),
        'MAPE':           round(mape, 2),
        'R2':             round(r2, 4),
        'SLA_actual_%':   round(sla_actual, 2),
        'SLA_pred_%':     round(sla_predicted, 2),
    }

---
## 4 · Baseline Models

Two baselines evaluated on the validation set before training LightGBM.

In [ ]:
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values

results = []

# ── Baseline 1: Global mean ───────────────────────────────────────────────
global_mean = y_train.mean()
pred_global = np.full(len(y_val), global_mean)
results.append(
    evaluate_model(y_val, pred_global, 'Baseline: Global Mean')
)

# ── Baseline 2: Territory × courier_flow segment median ──────────────────
seg_median = (
    df_train.groupby(
        ['territory', 'courier_flow'], observed=True
    )[TARGET]
    .median()
    .reset_index()
    .rename(columns={TARGET: 'seg_median'})
)
pred_seg = (
    df_val[['territory', 'courier_flow']]
    .merge(seg_median, on=['territory', 'courier_flow'], how='left')
    ['seg_median']
    .fillna(global_mean)
    .values
)
results.append(
    evaluate_model(
        y_val, pred_seg, 'Baseline: Territory×Flow Median'
    )
)

for r in results:
    print(
        f"{r['label']:<38} "
        f"MAE={r['MAE']:.2f}  "
        f"RMSE={r['RMSE']:.2f}  "
        f"MAPE={r['MAPE']:.1f}%  "
        f"R²={r['R2']:.4f}"
    )

---
## 5 · LightGBM Training

Hyperparameters follow notebook 04 precedent; `regression_l1` is
chosen because ATD residuals are heavy-tailed and MAE is the
primary operational metric.

In [ ]:
X_train = df_train[FEATURE_COLS]
X_val   = df_val[FEATURE_COLS]

# Align category levels: val must use same categories as train
for col in CAT_FEATURES:
    if col in X_train.columns:
        X_val[col] = pd.Categorical(
            X_val[col],
            categories=X_train[col].cat.categories,
        )

lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_FEATURES,
    free_raw_data=False,
)
lgb_val = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=CAT_FEATURES,
    reference=lgb_train,
    free_raw_data=False,
)

params = {
    'objective':        'regression_l1',
    'metric':           ['mae', 'rmse'],
    'learning_rate':    0.02,   # was 0.05 — lower lr lets model train longer
    'num_leaves':       255,    # was 127 — more capacity
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'verbose':          -1,
    'seed':             SEED,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=100, verbose=False),  # was 50
    lgb.log_evaluation(period=200),
]

print('Training LightGBM (regression_l1, early stopping on val MAE)...')
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=5000,   # was 2000
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

print(f'\nBest iteration : {model.best_iteration}')

---
## 6 · Evaluate on Validation Set

In [ ]:
pred_val = np.clip(
    model.predict(X_val, num_iteration=model.best_iteration),
    a_min=0, a_max=None,
)
result_lgbm = evaluate_model(y_val, pred_val, 'LightGBM')
results.append(result_lgbm)

# ── Comparison table ──────────────────────────────────────────────────────
comparison = pd.DataFrame(results).set_index('label')
print('\n── Validation Performance Comparison ──')
print(comparison.to_string())

best_baseline_mae = comparison[
    comparison.index.str.startswith('Baseline')
]['MAE'].min()
lgbm_mae = comparison.loc['LightGBM', 'MAE']
improvement = (best_baseline_mae - lgbm_mae) / best_baseline_mae * 100
print(
    f'\nLightGBM MAE improvement vs best baseline: {improvement:.1f}%'
)

---
## 7 · Feature Importance (quick view)

In [ ]:
imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain':    model.feature_importance('gain'),
}).sort_values('gain', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    imp['feature'][::-1],
    imp['gain'][::-1],
    color='steelblue',
    edgecolor='none',
)
ax.set_title('Top 20 Features — Gain', fontweight='bold')
ax.set_xlabel('Total Gain')
plt.tight_layout()
plt.show()

print('Top 10 features by gain:')
print(imp.head(10).to_string(index=False))

---
## 8 · Save Artifacts

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
joblib.dump(model, MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

# ── Metadata ──────────────────────────────────────────────────────────────
metadata = {
    'feature_cols':          FEATURE_COLS,
    'numeric_features':      NUMERIC_FEATURES,
    'cat_features':          CAT_FEATURES,
    'target':                TARGET,
    'split_date_test_start': manifest.get('split_date_test_start'),
    'split_date_val_start':  manifest.get('split_date_val_start'),
    'best_iteration':        int(model.best_iteration),
    'val_mae':               result_lgbm['MAE'],
    'val_rmse':              result_lgbm['RMSE'],
    'val_mape':              result_lgbm['MAPE'],
    'val_r2':                result_lgbm['R2'],
    'sla_threshold_min':     SLA_THRESHOLD_MIN,
}
with open(META_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, indent=2)
print(f'Metadata saved → {META_PATH}')

# ── Validation predictions ────────────────────────────────────────────────
val_preds_df = pd.DataFrame({
    'workflow_uuid': df_val['workflow_uuid'].values,
    'ATD':           y_val,
    'ATD_predicted': pred_val,
})
val_preds_df.to_parquet(VAL_PRED_PATH, index=False, engine='pyarrow')
print(f'Val predictions → {VAL_PRED_PATH}  ({len(val_preds_df):,} rows)')

print(
    f'\nVal MAE  = {result_lgbm["MAE"]:.3f} min '
    f'({improvement:.1f}% better than best baseline)'
)